# 03. 모델 활용 방안

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, json
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config as C
from src.metrics import K, daily_matrix
from src.viz import setup_plot_style, heatmap, DOW_KR

setup_plot_style()
pd.set_option("display.max_columns", 50)
DATA, OUT = ROOT / C.DATA_PATH, ROOT / C.OUTPUT_DIR

required = ["predictions_test.csv", "run_info.json"]
missing = [f for f in required if not (OUT / f).exists()]
if missing:
    raise FileNotFoundError(f"run.py 실행 결과가 없습니다: {missing}")

preds = pd.read_csv(OUT / "predictions_test.csv", parse_dates=["ts", "date"])
info = json.loads((OUT / "run_info.json").read_text(encoding="utf-8"))
selected = info["selected_model"]
print("GBM 백엔드:", info["gbm_backend"], "/ 선정 모델:", selected)

## 1. 영향 변수와 상호작용

최종 모델을 같은 설정으로 재학습하고 `run.py` 예측과의 일치를 확인한다. 순열 중요도와 PDP는 행 단위 트리 모델에 적용하며, 선정 모델이 시퀀스 모델이면 GBM으로 분석한다.

In [ ]:
from src.data import load_raw, clean
from src.features import to_long, build_features, feature_columns
from src.experiment import train_test, fit_predict_model
from src.models import make_models

hourly, _, _ = clean(load_raw(DATA))
feat = build_features(to_long(hourly))
tr, te = train_test(feat)
kind, model, lookback = make_models()[selected]
p, _ = fit_predict_model(model, lookback, feat, tr, te)
print(f"{selected}: run.py 예측과의 최대 차이 {float(np.abs(p - preds[selected].to_numpy()).max()):.2e}")

analysis_name = selected
if getattr(model, "needs_full_frame", False):
    analysis_name = "gbm"
    kind, model, lookback = make_models()[analysis_name]
    fit_predict_model(model, lookback, feat, tr, te)
cols = feature_columns(lookback=lookback)
print("분석 모델:", analysis_name, f"(입력 범위 {lookback}, 피처 {len(cols)}개)")

In [ ]:
from sklearn.inspection import permutation_importance, PartialDependenceDisplay

pi = permutation_importance(model, te[cols], te["kw"], scoring="neg_mean_absolute_error",
                            n_repeats=5, random_state=C.SEED)
imp = pd.Series(pi.importances_mean, index=cols).sort_values()
ax = imp.plot(kind="barh", figsize=(7, 6), color="#8aa")
ax.set_xlabel("순열 시 MAE 증가 (kW)")
plt.show()

In [ ]:
# 정수형 피처는 PDP 격자 생성 시 반올림 문제가 있어 실수형으로 변환한다(트리 예측에는 영향 없음).
X = te[cols].sample(min(1000, len(te)), random_state=C.SEED).astype(float)
one_d = [c for c in ["hour", "lag_7d", "lag_1d", "prev_day_max", "temp", "prod_h"] if c in cols][:3]
two_d = [pair for pair in [("hour", "dow"), ("hour", "prod_h"), ("hour", "prev_is_off_day"), ("hour", "lag_1d")]
         if all(c in cols for c in pair)][:2]

fig, axes = plt.subplots(1, len(one_d), figsize=(14, 3.5))
PartialDependenceDisplay.from_estimator(model, X, one_d, ax=axes)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, len(two_d), figsize=(13, 4.5))
PartialDependenceDisplay.from_estimator(model, X, two_d, ax=axes)
plt.tight_layout()
plt.show()

In [ ]:
try:
    import shap
    explainer = shap.TreeExplainer(model)
    sample = te[cols].sample(min(2000, len(te)), random_state=C.SEED)
    shap.summary_plot(explainer.shap_values(sample), sample, show=False)
    plt.show()
except ImportError:
    print("shap 미설치")
except Exception as e:
    print("SHAP 생략:", e)

## 2. 피크 시간대 예측 캘린더

일자·시간별로 예측 상위 10개 구간 수(0~4)를 표시하고, 실제 상위 10개 구간이 있는 시간을 ×로 표시한다.

In [ ]:
A = daily_matrix(preds, "kw")
P = daily_matrix(preds, selected)
top_a = np.argsort(-A.to_numpy(), axis=1, kind="stable")[:, :K] // 4
top_p = np.argsort(-P.to_numpy(), axis=1, kind="stable")[:, :K] // 4

cnt_p = np.zeros((len(P), 24))
cnt_a = np.zeros((len(A), 24))
np.add.at(cnt_p, (np.arange(len(P))[:, None].repeat(K, 1), top_p), 1)
np.add.at(cnt_a, (np.arange(len(A))[:, None].repeat(K, 1), top_a), 1)
labels = [f"{d:%m-%d} ({DOW_KR[d.dayofweek + 1]})" for d in P.index]
fig, ax = plt.subplots(figsize=(13, 8))
heatmap(pd.DataFrame(cnt_p, index=labels, columns=range(24)), ax=ax, cmap="Reds",
        cbar_label=f"예측 상위 {K}개 구간 수 ({selected})")
yy, xx = np.where(cnt_a > 0)
ax.scatter(xx, yy, marker="x", color="black", s=18, label=f"실제 상위 {K}개 포함 시간")
ax.set_xlabel("시간")
ax.legend(loc="upper right", fontsize=8)
plt.show()